In [1]:
import os

from dotenv import load_dotenv
from sqlitesearch import TextSearchIndex


load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY was not found. "
        "Check the .env file in your project folder."
    )

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

document_count = sqlite_index.count()

print(f"Documents in the database: {document_count}")

if document_count == 0:
    raise RuntimeError(
        "The database is empty. Run sqlite_ingest.py first."
    )

Documents in the database: 144


In [3]:
INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student, and your task is
to answer it.

Use the search function to find information in the course FAQ.

For the first search, use as many useful keywords from the user's
question as possible.

Make multiple searches when necessary. Analyze the results from
each search and perform additional searches using new keywords.

If a search returns poor results, consider whether the user's
question contains a spelling mistake. Try searching again with
corrected or alternative terms.

The question must be about the course or its logistics.
Do not answer off-topic questions.

If you cannot answer using the FAQ database, say that you don't know.
Do not answer using your own general knowledge.

At the end, ask whether the user wants to explore another
course-related area.
""".strip()


def search(query: str) -> list[dict]:
    """
    Search the LLM Zoomcamp FAQ database for entries matching
    the given query.
    """

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict={
            "question": 3.0,
            "section": 0.5,
        },
        filter_dict={
            "course": "llm-zoomcamp",
        },
    )

In [4]:
from toyaikit.tools import Tools


agent_tools = Tools()
agent_tools.add_tool(search)

generated_tools = agent_tools.get_tools()
generated_tools

[{'type': 'function',
  'name': 'search',
  'description': 'Search the LLM Zoomcamp FAQ database for entries matching\nthe given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [5]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    DisplayingRunnerCallback,
    OpenAIResponsesRunner,
)


chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(
    chat_interface
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini"
    ),
)

print("ToyAIKit runner created.")

ToyAIKit runner created.


In [6]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [7]:
print("Cost:")
print(result.cost)

print("\nNumber of messages:")
print(len(result.all_messages))

Cost:
CostInfo(input_cost=Decimal('0.00148575'), output_cost=Decimal('0.0009135'), total_cost=Decimal('0.00239925'))

Number of messages:
5


In [8]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student, and your task is\nto answer it.\n\nUse the search function to find information in the course FAQ.\n\nFor the first search, use as many useful keywords from the user's\nquestion as possible.\n\nMake multiple searches when necessary. Analyze the results from\neach search and perform additional searches using new keywords.\n\nIf a search returns poor results, consider whether the user's\nquestion contains a spelling mistake. Try searching again with\ncorrected or alternative terms.\n\nThe question must be about the course or its logistics.\nDo not answer off-topic questions.\n\nIf you cannot answer using the FAQ database, say that you don't know.\nDo not answer using your own general knowledge.\n\nAt the end, ask whether the user wants to explore another\ncourse-related area.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role

In [9]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received
